# Práctica 3. Detección y reconocimiento de formas

---

**Autores:**
- Alba Ramos Quintana
- Néstor Jesús Henríquez Medina

**Descripción:** Resolución de las tareas de la Práctica 3, incluyendo la identificación y clasificación de monedas para calcular cantidades de dinero, el procesamiento de imágenes de microplásticos mediante técnicas de umbralizado y el desarrollo de un clasificador basado en características geométricas y visuales.

---

## Tarea 1: Detección y reconocimiento de monedas

### Tarea 1.1: Cálculo del dinero en una imagen

- Determina la cantidad total de dinero presente en una imagen de monedas a partir de sus dimensiones. Para ello, establece una relación entre píxeles y milímetros utilizando como referencia una moneda de valor conocido y compara los diámetros medidos con las dimensiones reales de las monedas de euro.

In [88]:
%matplotlib qt

import cv2  
import numpy as np
import matplotlib.pyplot as plt
import csv

diametro_monedas = {
    2.00: 25.75,
    1.00: 23.25,
    0.50: 24.35,
    0.20: 22.25,
    0.10: 19.75,
    0.05: 21.25,
    0.02: 18.75,
    0.01: 16.25
}

def clasificar_moneda(diametro_medido, dicccionario):
    """
    Adivina de qué moneda se trata comparando su tamaño medido con las medidas oficiales.

    Calcula la diferencia entre el diámetro que hemos sacado en milímetros y los 
    tamaños reales del diccionario y escoge la opción que más se acerque.
    """
    moneda = None
    min_diferencia = float('inf')

    for valor, diametro in dicccionario.items():
        dif = abs(diametro - diametro_medido)
        if dif < min_diferencia:
            min_diferencia = dif
            moneda = valor
    return moneda

img = cv2.imread('Monedas.jpg') 
#img = cv2.imread('monedas2.jpg')
print(img.shape)

img_gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
cv2.imwrite('tarea_1/1_original_gris.jpg', img_gris)

img_suavizada = cv2.GaussianBlur(img_gris, (5, 5), 0)
cv2.imwrite('tarea_1/2_suavizada_gaussiana.jpg', img_suavizada)

img_mediana = cv2.medianBlur(img_gris, 5)
cv2.imwrite('tarea_1/2_suavizada_mediana.jpg', img_mediana)

th,img_th = cv2.threshold(img_gris,0,255,cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
print('Umbral Otsu ', th)
cv2.imwrite('tarea_1/3_binarizada.jpg', img_th)

contornos, _ = cv2.findContours(
    img_th, 
    cv2.RETR_EXTERNAL , 
    cv2.CHAIN_APPROX_SIMPLE
    )

print("Hay un total de", len(contornos), "contonos detectados")


# Fuente: https://hetpro-store.com/TUTORIALES/opencv-findcontours/?srsltid=AU7gw4Vh8qvHAzdz3LKVhhBp38z6-qQP5ekuIxHiYYR8xJRL_PztFQNX
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
cv2.drawContours(img_rgb, contornos, -1, (0,0,255), 3)

plt.plot()
plt.axis("off")
plt.imshow(img_rgb) 

# Fuente: https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.ginput.html
# Fuente: búsqueda en Google - "ginput() en ipynb"
# pip install PyQt5
pts = plt.ginput(1)

if len(pts) > 0:
    clic_x, clic_y = pts[0]
    print(f"Coordenada seleccionada: X= {clic_x:.2f}, Y= {clic_y:.2f}")

    contorno_ref = None
    for contorno in contornos:
        # Fuente: búsqueda en Google - "Averiguar punto dentro de un contorno con findContours()"
        ref = cv2.pointPolygonTest(contorno, (clic_x, clic_y), False)
        #print(ref)
        if ref >= 0:
            contorno_ref = contorno
            print("Esto es una moneda")
            break

    # Fuente: https://docs.opencv.org/4.13.0/dd/d49/tutorial_py_contour_features.html
    if contorno_ref is not None:
        _, radio_ref = cv2.minEnclosingCircle(contorno_ref)
        diametro_ref = radio_ref*2
        escala = diametro_monedas[1.00] / diametro_ref
        #print(escala)

        cv2.drawContours(img_rgb, [contorno_ref], -1, (0,255,0), 5)
        plt.imshow(img_rgb)
        plt.draw()

        total = 0
        diametro_min_px = 15/escala # min 15 mm
        for contorno in contornos:
            centro, radio = cv2.minEnclosingCircle(contorno)
            diametro_px = radio * 2
            
            # Si es muy pequeño, pasamos al siguiente contorno
            if diametro_px < diametro_min_px:
                continue
                
            diametro_mm = diametro_px * escala
            valor_moneda = clasificar_moneda(diametro_mm, diametro_monedas)
            
            total += valor_moneda
            
            texto = f"{valor_moneda}€"
            x, y = int(centro[0]), int(centro[1])
            cv2.putText(img_rgb, texto, (x - 20, y + 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)
        
        total_text = f"TOTAL = {total:.2f}€"
        tam_text, _ = cv2.getTextSize(total_text, cv2.FONT_HERSHEY_SIMPLEX, 1.2, 2)

        cv2.putText(img_rgb, total_text, (int((img.shape[1] - tam_text[0]) / 2), img.shape[0]), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 2, cv2.LINE_AA)

        print("\n==============")
        print(total_text)
        print("==============")

        img_final_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
        cv2.imwrite('tarea_1/4_resultado.jpg', img_final_bgr)
        plt.imshow(img_rgb)
        plt.draw()


(938, 473, 3)
Umbral Otsu  204.0
Hay un total de 50 contonos detectados
Coordenada seleccionada: X= 76.11, Y= 325.36
Esto es una moneda

TOTAL = 3.88€


---

### Tarea 1.2: Demostrador interactivo en tiempo real

- Diseña un demostrador que utilice la cámara para detectar las monedas presentes en una escena y determinar la cantidad de dinero total, sin utilizar modelos de aprendizaje profundo.

---

## Tarea 2: Segmentación y clasificación de microplásticos

- Aplica técnicas de umbralizado mediante OpenCV para segmentar muestras de microplásticos en imágenes reales y desarrolla un clasificador basado en características geométricas y visuales que permita distinguir los distintos tipos de partículas.

---